In [1]:
%load_ext autoreload
%autoreload 

In [2]:
#==========================================================
# Load Packages 
#==========================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
sys.path.append('/project/IZZY/MolRepres/Methods/')                 # Path where the model is located 
#from dig.threedgraph.dataset import QM93D                          # Dataset utilities for 3D molecular graphs (QM9 with 3D coordinates)
from torch_geometric.datasets import QM9                            # Loading the QM9 dataset from Torch-Geometric
from torch_geometric.loader import DataLoader                       # DataLoader to batch and shuffle molecular graph data
from torch.optim import Adam                                        # Optimizer (Adam) and learning rate scheduler (StepLR)
from torch.optim.lr_scheduler import StepLR                         # Progress bar for training/evaluation loops
from tqdm import tqdm                                               # Unit conversion constants (Hartree, eV, Bohr, Angstrom) from ASE
from ase.units import Hartree, eV, Bohr, Ang                        # TensorBoard writer for tracking metrics and visualizing training progress
from torch.utils.tensorboard import SummaryWriter                   # TensorBoard writer for tracking metrics and visualizing training progress 
                                  # 3D GNN model implementation (SphereNet) from DIG (Deep Graph Library for 3D Graphs)
from train_eval import train_epoch, evaluate                        # Importing the training and evaluating function
import torch_geometric
print(torch_geometric.__version__)
from torch_geometric.utils import to_dense_batch
from torch_scatter import scatter_mean
from torch_geometric.datasets import QM9
#from se3_transformer_pytorch import SE3Transformer
# from equiformer_pytorch import Equiformer
import equiformer_pytorch
from equiformer_pytorch import Equiformer
from torch_geometric.utils import to_dense_batch, to_dense_adj

2.6.1


In [3]:
#==========================================================
# GPU Device
#==========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # This line checks if a CUDA-enabled GPU is available. If yes, computations will be performed on the GPU for faster training. Otherwise, it falls back to using the CPU.
# Print which device is being used 
print("Using device:", device)

Using device: cuda


In [4]:
#==============================================================
# Load dataset and split
#==============================================================

# Load the QM9 dataset 
dataset = QM9(root='data/QM9')

# Count how many molecular graphs are in the dataset
num_mols = len(dataset)
print(f"Total QM9 molecules: {num_mols}")

# Define the desired split sizes for training, validation, and testing
train_size, valid_size = 110000, 10000
# Ensure the sum of train and validation sizes does not exceed dataset size
assert train_size + valid_size < num_mols, "Split sizes too large for dataset"

# Randomly shuffle all molecule indices
perm = torch.randperm(num_mols, generator=torch.Generator().manual_seed(42))

# Use the shuffled indices to create non-overlapping subsets
train_idx = perm[:train_size]
valid_idx = perm[train_size:train_size+valid_size]
test_idx  = perm[train_size+valid_size:]

# Build subsets based on the index splits
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset  = dataset[test_idx]

Total QM9 molecules: 130831


In [5]:
#======================================================================
# Define the QM9 Targets
#======================================================================
PROPERTY_NAMES = [
    'μ (D)', 'α (Ang³)', 'ε_HOMO (eV)', 'ε_LUMO (eV)', 'Δε (eV)', '⟨R²⟩ (Ang²)',
    'ZPVE (eV)', 'U₀ (eV)', 'U (eV)', 'H (eV)', 'G (eV)', 'c_v', 'U₀_atom',
    'U_atom', 'H_atom', 'G_atom', 'A (GHz)', 'B (GHz)', 'C (GHz)'
] 

# ======================================================================
# QM9 Unit Conversion Factors
# ======================================================================
def get_qm9_conversions_tensor(device):
    return torch.tensor([
        1.0, Bohr**3 / Ang**3, Hartree / eV, Hartree / eV, Hartree / eV,
        Bohr**2 / Ang**2, Hartree / eV, Hartree / eV, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, 1.0
    ], dtype=torch.float, device=device)

In [6]:
#========================================================================
# Normalize all QM9 targets
#========================================================================
y_raw_all = dataset.data.y.clone().cpu()
conversions_cpu = get_qm9_conversions_tensor('cpu')
y_conv_all = y_raw_all * conversions_cpu.unsqueeze(0)

norm_stats = {'mean': [], 'std': []}
y_norm_all = torch.zeros_like(y_conv_all)

for i in range(y_conv_all.shape[1]):
    train_y_cpu = y_conv_all[train_idx.cpu(), i]
    mean_i = float(train_y_cpu.mean().item())
    std_i = float(train_y_cpu.std().item()) if train_y_cpu.std().item() != 0 else 1.0
    y_norm_all[:, i] = (y_conv_all[:, i] - mean_i) / std_i
    norm_stats['mean'].append(mean_i)
    norm_stats['std'].append(std_i)

dataset.data.y = y_norm_all.to(torch.float)
print("Normalization complete for all targets.")

Normalization complete for all targets.


/home/ismail/dig_envi/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [26]:
from rdkit import Chem

from rdkit import Chem
from rdkit.Chem import AllChem

smiles = "CC(=O)O"  # acetic acid (has carbonyl + carboxylic acid motifs)
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# generate 3D coordinates (needed for .GetConformer())
AllChem.EmbedMolecule(mol, AllChem.ETKDG())
AllChem.UFFOptimizeMolecule(mol)

print(mol.GetNumAtoms())

FG_SMARTS = {
    "hydroxyl": "[OX2H]",                # -OH
    "carbonyl": "[CX3]=[OX1]",           # C=O
    "carboxylic_acid": "C(=O)[OX2H1]",   # -C(=O)OH
    "ester": "C(=O)O[#6]",               # -C(=O)O-
    "amine": "[NX3;H2,H1,H0;!$(NC=O)]",  # amines (exclude amides)
    "amide": "C(=O)N",                   # amide
    "halogen": "[F,Cl,Br,I]",            # halogens
    "aromatic_ring": "a1aaaaa1",         # simple aromatic ring
}
FG_TYPE_TO_ID = {name: i for i, name in enumerate(sorted(FG_SMARTS.keys()))}


8


In [27]:
import numpy as np

def rdkit_atom_features(mol: Chem.Mol) -> np.ndarray:
    """
    Minimal atom features example (you probably already have your own).
    Return shape: (N, n_token)
    """
    # Example: one-hot by atomic number bucket (QM9-like simplified).
    # Replace with your real n_token=11 feature logic.
    zs = [a.GetAtomicNum() for a in mol.GetAtoms()]
    allowed = [1, 6, 7, 8, 9]  # H, C, N, O, F
    n_token = 11
    x = np.zeros((mol.GetNumAtoms(), n_token), dtype=np.float32)
    for i, z in enumerate(zs):
        if z in allowed:
            x[i, allowed.index(z)] = 1.0
        else:
            x[i, len(allowed)] = 1.0  # "other"
    return x

def rdkit_coords(mol: Chem.Mol) -> np.ndarray:
    conf = mol.GetConformer()
    n = mol.GetNumAtoms()
    pos = np.zeros((n, 3), dtype=np.float32)
    for i in range(n):
        p = conf.GetAtomPosition(i)
        pos[i] = (p.x, p.y, p.z)
    return pos

def rdkit_bond_edge_index(mol: Chem.Mol) -> np.ndarray:
    edges = []
    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()
        edges.append((i, j))
        edges.append((j, i))
    if len(edges) == 0:
        return np.zeros((2, 0), dtype=np.int64)
    return np.array(edges, dtype=np.int64).T


In [28]:
def build_fg_memberships(mol: Chem.Mol, fg_smarts: dict):
    """
    Returns:
      fg_type_ids: (num_fg,) int64
      fg_atom_sets: list[list[int]] length num_fg
    """
    fg_type_ids = []
    fg_atom_sets = []

    for fg_name, smarts in fg_smarts.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt is None:
            continue

        matches = mol.GetSubstructMatches(patt, uniquify=True)
        for match in matches:
            aset = sorted(set(match))
            if len(aset) == 0:
                continue
            fg_type_ids.append(FG_TYPE_TO_ID[fg_name])
            fg_atom_sets.append(aset)

    return np.array(fg_type_ids, dtype=np.int64), fg_atom_sets


In [29]:
def compute_fg_centers(atom_pos: np.ndarray, fg_atom_sets: list[list[int]]) -> np.ndarray:
    centers = []
    for aset in fg_atom_sets:
        pts = atom_pos[np.array(aset, dtype=np.int64)]
        centers.append(pts.mean(axis=0))
    if len(centers) == 0:
        return np.zeros((0, 3), dtype=np.float32)
    return np.stack(centers, axis=0).astype(np.float32)


In [30]:
def build_fg_edges_from_bonds(mol: Chem.Mol, fg_atom_sets: list[list[int]]) -> np.ndarray:
    num_fg = len(fg_atom_sets)
    if num_fg == 0:
        return np.zeros((2, 0), dtype=np.int64)

    atom_to_fgs = {}
    for fg_id, aset in enumerate(fg_atom_sets):
        for a in aset:
            atom_to_fgs.setdefault(a, []).append(fg_id)

    edges = set()
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        fgs_a = atom_to_fgs.get(a, [])
        fgs_b = atom_to_fgs.get(b, [])
        for ga in fgs_a:
            for gb in fgs_b:
                if ga != gb:
                    edges.add((ga, gb))
                    edges.add((gb, ga))

    if not edges:
        return np.zeros((2, 0), dtype=np.int64)
    return np.array(list(edges), dtype=np.int64).T


In [31]:
def build_fg_edges_knn(fg_pos: np.ndarray, k: int = 2) -> np.ndarray:
    n = fg_pos.shape[0]
    if n <= 1 or k <= 0:
        return np.zeros((2, 0), dtype=np.int64)

    d2 = ((fg_pos[:, None, :] - fg_pos[None, :, :]) ** 2).sum(axis=-1)
    np.fill_diagonal(d2, np.inf)

    edges = []
    for i in range(n):
        nn = np.argsort(d2[i])[:min(k, n - 1)]
        for j in nn:
            edges.append((i, int(j)))

    if not edges:
        return np.zeros((2, 0), dtype=np.int64)
    return np.array(edges, dtype=np.int64).T


In [32]:
def merge_edge_index(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if a.size == 0:
        return b
    if b.size == 0:
        return a
    edges = set(map(tuple, a.T.tolist()))
    edges |= set(map(tuple, b.T.tolist()))
    return np.array(list(edges), dtype=np.int64).T


In [33]:
def build_membership_edges(num_atoms: int, fg_atom_sets: list[list[int]]) -> tuple[np.ndarray, np.ndarray]:
    atom_to_fg = []
    fg_to_atom = []
    for fg_id, aset in enumerate(fg_atom_sets):
        for a in aset:
            atom_to_fg.append((a, fg_id))
            fg_to_atom.append((fg_id, a))

    if not atom_to_fg:
        return (np.zeros((2, 0), dtype=np.int64), np.zeros((2, 0), dtype=np.int64))

    return (np.array(atom_to_fg, dtype=np.int64).T,
            np.array(fg_to_atom, dtype=np.int64).T)


In [34]:
import torch
from torch_geometric.data import HeteroData

def mol_to_heterodata(mol: Chem.Mol, knn_k: int = 2) -> HeteroData:
    # atom-level
    atom_x = rdkit_atom_features(mol)          # (N, n_token)
    atom_pos = rdkit_coords(mol)               # (N, 3)
    atom_edge_index = rdkit_bond_edge_index(mol)  # (2, E_atom)

    # functional groups
    fg_type_ids, fg_atom_sets = build_fg_memberships(mol, FG_SMARTS)  # (G,), list[atoms]
    fg_pos = compute_fg_centers(atom_pos, fg_atom_sets)               # (G, 3)
    fg_edge_bond = build_fg_edges_from_bonds(mol, fg_atom_sets)       # (2, E_fg_bond)
    fg_edge_knn = build_fg_edges_knn(fg_pos, k=knn_k)                 # (2, E_fg_knn)
    fg_edge_index = merge_edge_index(fg_edge_bond, fg_edge_knn)       # (2, E_fg)
    atom_to_fg, fg_to_atom = build_membership_edges(mol.GetNumAtoms(), fg_atom_sets)

    data = HeteroData()

    # nodes
    data["atom"].x = torch.from_numpy(atom_x)         # float32
    data["atom"].pos = torch.from_numpy(atom_pos)     # float32
    data["fg"].type_id = torch.from_numpy(fg_type_ids)  # int64
    data["fg"].pos = torch.from_numpy(fg_pos)           # float32

    # edges
    data["atom", "bond", "atom"].edge_index = torch.from_numpy(atom_edge_index)
    data["fg", "link", "fg"].edge_index = torch.from_numpy(fg_edge_index)
    data["atom", "in_fg", "fg"].edge_index = torch.from_numpy(atom_to_fg)
    data["fg", "has_atom", "atom"].edge_index = torch.from_numpy(fg_to_atom)

    return data


In [35]:
data = mol_to_heterodata(mol, knn_k=2)
print(type(data))
print(data.node_types)
print(data.edge_types)
print(data)



<class 'torch_geometric.data.hetero_data.HeteroData'>
['atom', 'fg']
[('atom', 'bond', 'atom'), ('fg', 'link', 'fg'), ('atom', 'in_fg', 'fg'), ('fg', 'has_atom', 'atom')]
HeteroData(
  atom={
    x=[8, 11],
    pos=[8, 3],
  },
  fg={
    type_id=[3],
    pos=[3, 3],
  },
  (atom, bond, atom)={ edge_index=[2, 14] },
  (fg, link, fg)={ edge_index=[2, 6] },
  (atom, in_fg, fg)={ edge_index=[2, 6] },
  (fg, has_atom, atom)={ edge_index=[2, 6] }
)


In [36]:
sample = dataset[0]
print(sample)
mol.GetConformer()


Data(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], z=[5], smiles='[H]C([H])([H])[H]', name='gdb_1', idx=[1])


In [37]:
mol.GetConformer()


In [38]:
print(mol.GetConformer().Is3D())  # should be True


True


In [39]:
print(data.node_types)
print(data.edge_types)


['atom', 'fg']
[('atom', 'bond', 'atom'), ('fg', 'link', 'fg'), ('atom', 'in_fg', 'fg'), ('fg', 'has_atom', 'atom')]


In [41]:
print(data)

HeteroData(
  atom={
    x=[8, 11],
    pos=[8, 3],
  },
  fg={
    type_id=[3],
    pos=[3, 3],
  },
  (atom, bond, atom)={ edge_index=[2, 14] },
  (fg, link, fg)={ edge_index=[2, 6] },
  (atom, in_fg, fg)={ edge_index=[2, 6] },
  (fg, has_atom, atom)={ edge_index=[2, 6] }
)


In [42]:
import torch
import torch.nn as nn

from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.nn import GATConv
from torch_geometric.utils import scatter  # scatter(src, index, dim=0, reduce="mean")


class EquiformerWithFG(nn.Module):
    def __init__(
        self,
        equiformer_ctor,          # pass your Equiformer(...) constructor or class
        n_token=11,
        n_out=19,
        hidden_dim=128,
        num_fg_types=8,           # len(FG_SMARTS)
        fg_dim=128,
        fg_gnn_layers=2
    ):
        super().__init__()

        # Atom input embedding (scalars)
        self.atom_embed = nn.Linear(n_token, hidden_dim)

        # Equiformer v1 core (your exact config)
        self.equiformer = equiformer_ctor(
            dim=hidden_dim,
            dim_in=hidden_dim,
            input_degrees=1,
            num_degrees=2,
            heads=4,
            dim_head=hidden_dim // 4,
            depth=8,
            attend_sparse_neighbors=True,
            num_neighbors=8,
            num_adj_degrees_embed=2,
            max_sparse_neighbors=16,
            valid_radius=5.0,
            reduce_dim_out=False,
            attend_self=True,
            l2_dist_attention=False
        )

        # FG type embedding (scalar FG features)
        self.fg_embed = nn.Embedding(num_fg_types, fg_dim)

        # Optional projection so FG and atom hidden dims match
        self.fg_proj = nn.Linear(fg_dim, hidden_dim)

        # FG-level GNN (scalar graph, simple & strong baseline)
        self.fg_gnns = nn.ModuleList([
            GATConv(hidden_dim, hidden_dim, heads=1, concat=False)
            for _ in range(fg_gnn_layers)
        ])
        self.fg_act = nn.ReLU()

        # Final regression head (molecule-level)
        self.head = nn.Linear(hidden_dim, n_out)

    def _extract_type0(self, out):
        # same defensive logic you used
        if hasattr(out, "type0"):
            return out.type0
        if isinstance(out, dict):
            return out.get(0, next(iter(out.values())))
        if isinstance(out, (list, tuple)):
            return out[0]
        return out

    def forward(self, data):
        """
        data: a batched HeteroData with:
          data["atom"].x, data["atom"].pos, data["atom"].batch
          data["atom","bond","atom"].edge_index
          data["fg"].type_id, data["fg"].batch (may be empty)
          data["fg","link","fg"].edge_index
          data["atom","in_fg","fg"].edge_index
          data["fg","has_atom","atom"].edge_index
        """

        # -------------------------
        # A) Atom branch: Equiformer
        # -------------------------
        atom_x = data["atom"].x
        atom_pos = data["atom"].pos
        atom_batch = data["atom"].batch
        atom_edge_index = data["atom", "bond", "atom"].edge_index

        atom_x = self.atom_embed(atom_x)  # (total_atoms, hidden_dim)

        # Dense batching for Equiformer
        x_dense, mask = to_dense_batch(atom_x, atom_batch)         # (B, N, F), (B, N)
        pos_dense, _ = to_dense_batch(atom_pos, atom_batch)        # (B, N, 3)

        # Bond adjacency mask (B, N, N)
        adj_mat = to_dense_adj(atom_edge_index, batch=atom_batch).bool()

        out = self.equiformer(x_dense, pos_dense, mask=mask, adj_mat=adj_mat)

        x0_dense = self._extract_type0(out)  # expect (B, N, F) or (B, F)
        if x0_dense.ndim == 2:
            # already pooled (B,F) — rare; handle anyway
            mol_repr = x0_dense
            return self.head(mol_repr)

        if x0_dense.ndim != 3:
            raise ValueError(f"Unexpected Equiformer output shape: {x0_dense.shape}")

        # Flatten back to per-atom features (total_atoms, F) in the same order as input atoms
        atom_h = x0_dense[mask]  # (total_atoms, hidden_dim)

        # -------------------------
        # B) FG branch: build/update FG features
        # -------------------------
        # Some molecules may have 0 FGs; handle empty case robustly
        num_fg = data["fg"].type_id.size(0)

        if num_fg > 0:
            fg_type_id = data["fg"].type_id
            fg_h = self.fg_proj(self.fg_embed(fg_type_id))  # (total_fg, hidden_dim)

            # -------------------------
            # C) Atom -> FG pooling (membership edges)
            # -------------------------
            a2g = data["atom", "in_fg", "fg"].edge_index  # [2, E]
            if a2g.numel() > 0:
                src_atom = a2g[0]
                dst_fg = a2g[1]
                pooled = scatter(atom_h[src_atom], dst_fg, dim=0, dim_size=num_fg, reduce="mean")
                fg_h = fg_h + pooled

            # -------------------------
            # D) FG message passing
            # -------------------------
            fg_edge_index = data["fg", "link", "fg"].edge_index
            if fg_edge_index.numel() > 0:
                for conv in self.fg_gnns:
                    fg_h = conv(fg_h, fg_edge_index)
                    fg_h = self.fg_act(fg_h)

            # -------------------------
            # E) FG -> Atom broadcast (membership reverse edges)
            # -------------------------
            g2a = data["fg", "has_atom", "atom"].edge_index
            if g2a.numel() > 0:
                src_fg = g2a[0]
                dst_atom = g2a[1]
                fg_msg = scatter(fg_h[src_fg], dst_atom, dim=0, dim_size=atom_h.size(0), reduce="mean")
                atom_h = atom_h + fg_msg

        # -------------------------
        # F) Molecule pooling + regression
        # -------------------------
        # Pool per atom to per molecule (B, hidden_dim)
        mol_repr = scatter(atom_h, atom_batch, dim=0, reduce="mean")

        return self.head(mol_repr)


In [49]:
data = mol_to_heterodata(mol)
dataset = [data]   # list with ONE HeteroData
print(len(dataset))


1


In [50]:
from torch_geometric.loader import DataLoader

loader = DataLoader(dataset, batch_size=1, shuffle=False)


In [52]:
#==============================================================================================
# Trainig Loop
#==============================================================================================

def main():
    #------------------------------
    # Hyperparameters
    #------------------------------
    epochs = 10                           # number of training epochs
    batch_size = 128                      # batch size for training
    vt_batch_size = 256                  # batch size for validation
    lr = 3e-4                             # learning rate
    lr_decay_step = 50                    # steps after which LR is decayed
    lr_decay_factor = 0.5                 # factor to decay learning rate
    weight_decay = 1e-4                   # L2 regularization
    save_dir = 'checkpoints_EquiformerFG'    # directory to save model checkpoints 
    log_dir = 'logs_EquiformerFG'            # TensorBoard logs directory

    #Create directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # TensorBoard write for visualization of metrics
    writer = SummaryWriter(log_dir=log_dir)

    #-----------------------------
    # Dataloaders 
    #-----------------------------

    # Shuffled batches for training; sequential batches for validation/testing
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(valid_dataset, batch_size=vt_batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=vt_batch_size, shuffle=False)

    #----------------------------------------
    # Model
    #----------------------------------------
    # SphereNet model
    model = EquiformerWithFG().to(device)   

    #----------------------------------------
    # Optimizer and Scheduler
    #----------------------------------------
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Track best validation/test performance
    best_mean_val=float('inf')
    best_val = [float('inf')] * len(PROPERTY_NAMES)
    best_test = [float('inf')] * len(PROPERTY_NAMES)

    # Print total number of trainabel parameters and target property
    print("#params:", sum(p.numel() for p in model.parameters()))
    print("Training for all 19 targets") 
    
    #--------------------------------------------
    # Training Loop
    #--------------------------------------------
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        #Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, device, accum_steps = 4)

        # Evaluate on validation and test sets (MAE in original units)
        val_mae = evaluate(model, val_loader, device, norm_stats['mean'], norm_stats['std'])
        test_mae = evaluate(model, test_loader, device, norm_stats['mean'], norm_stats['std'])

        # Print epoch metrics
        print(f"Train loss (MSE): {train_loss:.6f}")
        for i, prop in enumerate(PROPERTY_NAMES):
            print(f"  {prop:15s} | Val MAE: {val_mae[i]:.6f} | Test MAE: {test_mae[i]:.6f}")
            writer.add_scalar(f'val_mae/{prop}', val_mae[i], epoch)
            writer.add_scalar(f'test_mae/{prop}', test_mae[i], epoch)
        #---------------------------------------------
        # Save checkpoint if validation improves 
        #---------------------------------------------
        mean_val_mae = sum(val_mae) / len(val_mae)
        
        # If mean validation MAE improved → save one best model
        if mean_val_mae < best_mean_val:
            best_mean_val = mean_val_mae
            best_val = val_mae
            best_test = test_mae
        
            save_path = os.path.join(save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val': best_val,
                'best_test': best_test,
                'best_mean_val': best_mean_val
            }, save_path)
        
            print(f" Saved best overall model (mean validation MAE improved to {best_mean_val:.4f})")

        # Update learning rate according to scheduler
        scheduler.step()
    # Close TensorBoard writer
    writer.close()
    print("\nFinished training.")
    print("Best validation and test MAEs per property:")
    for prop, v, t in zip(PROPERTY_NAMES, best_val, best_test):
        print(f"  {prop:15s} | Best validation MAE: {v:.6f} | Test MAE at best val: {t:.6f}")

    #print("Best validation MAE:", best_val)
    #print("Test MAE at best val:", best_test)

# Run the training loop
if __name__ == "__main__":
    main()

TypeError: __init__() missing 1 required positional argument: 'equiformer_ctor'

In [51]:
for data in loader:
    data = data.to(device)
    pred = model(data)
    loss = criterion(pred, data.y)


NameError: name 'model' is not defined

In [45]:
print(type(batch))



NameError: name 'batch' is not defined